In [1]:
# =========================================================
# HBOS (Histogram-Based Outlier Score) - 이상 탐지 실험
# 데이터셋: NSL-KDD, UNSW-NB15
# 전처리  : MinMax / Quantile
# 임계값  : Bootstrap (α=0.10, α=0.15)
# 범주형  : 히스토그램 밀도 추정 (별도 인코딩 없음)
# =========================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix
)

# =========================================================
# 경로 설정
# =========================================================
DATA_DIR = "./data/"

NSL_CAT  = (
    "protocol_type", "service", "flag",
    "land", "logged_in", "is_guest_login",
    "is_host_login", "root_shell", "su_attempted",
)
UNSW_CAT = (
    "proto", "service", "state",
    "is_sm_ips_ports", "is_ftp_login",
)

# =========================================================
# HBOS 모델
# =========================================================
class HBOS:
    """
    수치형 : 등폭 히스토그램 밀도 추정  fhat = n_bin / (n * w_bin)
    범주형 : 상대 빈도                  fhat = n_cat / n
    이상 점수 : S(x) = -sum_j log(fhat_j(x_j))
    """
    def __init__(self, n_bins=10, eps=1e-300):
        self.n_bins        = int(n_bins)
        self.eps           = float(eps)
        self.feature_info_ = []
        self.n_samples_    = None

    def fit(self, X_num, X_cat=None):
        X_num = np.asarray(X_num)
        n     = X_num.shape[0]
        self.n_samples_    = n
        self.feature_info_ = []

        for j in range(X_num.shape[1]):
            col    = X_num[:, j]
            counts, bin_edges = np.histogram(col, bins=self.n_bins, density=False)
            widths    = np.diff(bin_edges)
            densities = counts / (n * widths)
            self.feature_info_.append(("num", (bin_edges, densities)))

        if X_cat is not None:
            X_cat = np.asarray(X_cat)
            for j in range(X_cat.shape[1]):
                col  = X_cat[:, j]
                vals, counts = np.unique(col, return_counts=True)
                freq = dict(zip(vals.tolist(), (counts / n).tolist()))
                self.feature_info_.append(("cat", (freq,)))
        return self

    def score_samples(self, X_num, X_cat=None):
        X_num   = np.asarray(X_num)
        n_q     = X_num.shape[0]
        scores  = np.zeros(n_q, dtype=np.float64)
        num_idx = cat_idx = 0

        if X_cat is not None:
            X_cat = np.asarray(X_cat)

        for ftype, data in self.feature_info_:
            if ftype == "num":
                bin_edges, densities = data
                col     = X_num[:, num_idx]
                bin_idx = np.searchsorted(bin_edges[1:-1], col, side="right")
                bin_idx = np.clip(bin_idx, 0, len(densities) - 1)
                fhat    = np.maximum(densities[bin_idx], self.eps)
                scores += -np.log(fhat)
                num_idx += 1
            elif ftype == "cat":
                (freq,) = data
                col     = X_cat[:, cat_idx]
                fhat    = np.maximum(
                    np.array([freq.get(v, 0.0) for v in col], dtype=np.float64),
                    self.eps
                )
                scores += -np.log(fhat)
                cat_idx += 1
        return scores

# =========================================================
# 특징 행렬 구성 헬퍼
# =========================================================
def build_X(df, cat_cols, class_col):
    X       = df.drop(columns=[class_col], errors="ignore").copy()
    cat_eff = [c for c in cat_cols if c in X.columns]
    num_eff = [c for c in X.columns
               if c not in cat_eff and pd.api.types.is_numeric_dtype(X[c])]
    X_num   = X[num_eff].to_numpy(dtype=np.float64) if num_eff else np.empty((len(X), 0))
    X_cat   = X[cat_eff].to_numpy() if cat_eff else None
    return X_num, X_cat

def make_finite(s):
    s   = np.asarray(s, dtype=np.float64).copy()
    fin = np.isfinite(s)
    if fin.any():
        s[~fin] = np.max(s[fin]) * 1.5
    return s

# =========================================================
# Bootstrap 임계값
# =========================================================
def bootstrap_threshold(scores, percentiles=range(0, 101), B=500, seed=42):
    scores = np.asarray(scores).ravel()
    rng    = np.random.default_rng(seed)
    n      = len(scores)
    boot   = {p: np.empty(B, dtype=np.float32) for p in percentiles}
    for b in range(B):
        sample = rng.choice(scores, size=n, replace=True)
        for p in percentiles:
            boot[p][b] = np.percentile(sample, p)
    df = pd.DataFrame(
        {p: float(np.median(boot[p])) for p in percentiles}.items(),
        columns=["Percentile", "Threshold"]
    ).set_index("Percentile")
    return df

# =========================================================
# 성능 평가
# =========================================================
def evaluate(test_scores, y_true, thresholds_dict):
    try:
        auc = roc_auc_score(y_true, test_scores)
    except ValueError:
        auc = np.nan

    rows = []
    for alpha, T in thresholds_dict.items():
        y_pred = (test_scores >= T).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        rows.append({
            "α":           alpha,
            "Precision":   precision_score(y_true, y_pred, zero_division=0),
            "Recall":      recall_score(y_true, y_pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "F1-score":    f1_score(y_true, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(y_true, y_pred),
            "AUC":         auc,
        })
    return pd.DataFrame(rows).sort_values("α").reset_index(drop=True)

# =========================================================
# 파이프라인
# =========================================================
def run_pipeline(train_df, valid_df, test_df,
                 cat_cols, class_col, normal_name,
                 n_bins=10, seed=42):

    Xnum_tr, Xcat_tr = build_X(train_df, cat_cols, class_col)
    Xnum_va, Xcat_va = build_X(valid_df, cat_cols, class_col)
    Xnum_te, Xcat_te = build_X(test_df,  cat_cols, class_col)

    scorer = HBOS(n_bins=n_bins).fit(Xnum_tr, Xcat_tr)

    valid_scores = make_finite(scorer.score_samples(Xnum_va, Xcat_va))
    df_thr       = bootstrap_threshold(valid_scores, seed=seed)

    # α=0.10 → P90, α=0.15 → P85
    thresholds = {0.10: float(df_thr.loc[90, "Threshold"]),
                  0.15: float(df_thr.loc[85, "Threshold"])}

    test_scores = make_finite(scorer.score_samples(Xnum_te, Xcat_te))
    y_true      = (test_df[class_col] != normal_name).astype(int).values

    return evaluate(test_scores, y_true, thresholds)

# =========================================================
# 결과 출력
# =========================================================
def print_results(dataset_name, df_mm, df_qt):
    COLS    = ["Precision", "Recall", "Specificity", "F1-score", "Accuracy", "AUC"]
    COLS_KR = ["정밀도",    "민감도",  "특이도",      "F1 점수",  "정확도",   "AUC"]
    W       = 78

    print("=" * W)
    print(f" {dataset_name}")
    print("=" * W)
    print(f"  {'':22s}" + "".join(f"{k:>8}" for k in COLS_KR))
    print("-" * W)

    for scaler, df in [("MinMax", df_mm), ("Quantile", df_qt)]:
        for _, row in df.iterrows():
            label = f"  {scaler:<10} α={row['α']:.2f}"
            vals  = "".join(f"{row[c]:>8.2f}" for c in COLS)
            print(f"{label:<26}{vals}")
        print("-" * W)
    print()

# =========================================================
# 실험 실행
# =========================================================
if __name__ == "__main__":

    # ── NSL-KDD ──────────────────────────────────────────
    train_nsl_mm = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_train_normal_80.csv")
    valid_nsl_mm = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_train_normal_20.csv")
    test_nsl_mm  = pd.read_csv(DATA_DIR + "NSL_KDD_MinMax_test.csv")

    train_nsl_qt = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_train_normal_80.csv")
    valid_nsl_qt = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_train_normal_20.csv")
    test_nsl_qt  = pd.read_csv(DATA_DIR + "NSL_KDD_Quantile_test.csv")

    res_nsl_mm = run_pipeline(train_nsl_mm, valid_nsl_mm, test_nsl_mm,
                              cat_cols=NSL_CAT, class_col="class", normal_name="normal")
    res_nsl_qt = run_pipeline(train_nsl_qt, valid_nsl_qt, test_nsl_qt,
                              cat_cols=NSL_CAT, class_col="class", normal_name="normal")

    # ── UNSW-NB15 ────────────────────────────────────────
    train_unsw_mm = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_train_normal_80.csv")
    valid_unsw_mm = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_train_normal_20.csv")
    test_unsw_mm  = pd.read_csv(DATA_DIR + "UNSW_NB15_MinMax_test.csv")

    train_unsw_qt = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_train_normal_80.csv")
    valid_unsw_qt = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_train_normal_20.csv")
    test_unsw_qt  = pd.read_csv(DATA_DIR + "UNSW_NB15_Quantile_test.csv")

    res_unsw_mm = run_pipeline(train_unsw_mm, valid_unsw_mm, test_unsw_mm,
                               cat_cols=UNSW_CAT, class_col="label", normal_name=0)
    res_unsw_qt = run_pipeline(train_unsw_qt, valid_unsw_qt, test_unsw_qt,
                               cat_cols=UNSW_CAT, class_col="label", normal_name=0)

    # ── 출력 ─────────────────────────────────────────────
    print_results("NSL-KDD",   res_nsl_mm,  res_nsl_qt)
    print_results("UNSW-NB15", res_unsw_mm, res_unsw_qt)

 NSL-KDD
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.92    0.71    0.92    0.80    0.80    0.92
  MinMax     α=0.15           0.92    0.76    0.91    0.83    0.82    0.92
------------------------------------------------------------------------------
  Quantile   α=0.10           0.96    0.75    0.96    0.84    0.84    0.95
  Quantile   α=0.15           0.94    0.79    0.94    0.86    0.85    0.95
------------------------------------------------------------------------------

 UNSW-NB15
                             정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------------
  MinMax     α=0.10           0.84    0.53    0.87    0.65    0.68    0.78
  MinMax     α=0.15           0.78    0.59    0.80    0.67    0.68    0.78
-------------------------------------------------------------------